# ETL ITAM — Inventario Hoteles → Data Warehouse
### Dry Run v2 — Columnas exactas del schema Prisma

Genera 7 CSVs que representan exactamente cómo quedarían las tablas en SQL Server.

| Orden | Tabla | Por qué ese orden |
|-------|-------|-------------------|
| 1 | `AssetState` | Sin FK, es catálogo base |
| 2 | `Company` | Sin FK |
| 3 | `Vendor` | Sin FK |
| 4 | `ProductType` | Sin FK |
| 5 | `Site` | Depende de Company |
| 6 | `Asset` | Depende de Company, Site, Vendor, ProductType, AssetState |
| 7 | `AssetDetail` | Depende de Asset |

## ⚙️ Configuración

In [ ]:
CSV_PATH   = '/content/drive/MyDrive/proyecto/TIENDAS ACTUALIZACIONES.csv'
OUTPUT_DIR = '/content/drive/MyDrive/proyecto/etl_output/'

# ID del usuario sistema que hace la carga (LastUpdateBy en AssetDetail)
# Debe existir en la tabla User de tu BD antes de correr el Load real
SYSTEM_USER_ID = 1

# Nombre del estado por defecto para activos recién cargados
# Se creará en la tabla AssetState si no existe
DEFAULT_ASSET_STATE_NAME = 'Activo'

## 📦 Imports

In [ ]:
import pandas as pd
import re
import os
from datetime import datetime

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)

## 1️⃣ Montar Drive y cargar CSV

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
raw_df = pd.read_csv(CSV_PATH, encoding='latin1')
print(f'✅ Filas cargadas  : {len(raw_df)}')
print(f'✅ Columnas        : {list(raw_df.columns)}')
display(raw_df.head(5))

## 2️⃣ Limpieza general
- Mayúsculas en todas las columnas de texto
- Corrige carácter corrupto `RECEPTOR\x8dA`
- Elimina símbolos que no sean letra, número o espacio

In [ ]:
def limpieza(texto):
    if pd.isna(texto) or not isinstance(texto, str):
        return texto
    texto = texto.upper()
    texto = texto.replace('RECEPTOR\x8dA', 'RECEPTORIA')
    texto = re.sub(r'[^A-Z0-9\s]', '', texto)
    return texto.strip()

util_df = raw_df.copy()
for col in util_df.columns:
    if util_df[col].dtype == 'object':
        util_df[col] = util_df[col].apply(limpieza)

print('✅ Limpieza aplicada')
display(util_df.head(5))

## 3️⃣ Separar HOTEL y TIPO_EQUIPO por empresa
Usa la primera palabra clave encontrada en TIENDAS como separador:
```
MOON GRAND JOY CAJA  →  HOTEL=MOON GRAND  |  TIPO_EQUIPO=JOY
```

In [ ]:
EMPRESA_CONFIG = {
    'ISLA17':    r'(TAB|JOY|CORONA|MULTI|PUEBLITO|RECEPTORA|AMIANI|AQUA PARK|PROSHOP|PIER 27|PIER27|CORAZON)',
    'HSPRO':     r'(TAB|JOY|RECEPTOR|AMIANI|PIER27|TABAQUERIA|BOUTIQUE|VIVID|LOBBY|WHITESANDS)',
    'PTOARENAS': r'(TAB|JOY|CORONA|MULTI|PUEBLITO|RECEPTORIA|AMIANI|AQUA PARK|PROSHOP|PIER 27|PIER27|CORAZON|VIVA|MARINI|SUNGLASSCHIC|ALBERCA|MUL|RECEPTOR)',
    'PTOHS':     r'(TAB|JOY|CORONA|MULTI|PUEBLITO|RECEPTORIA|AMIANI|AQUA PARK|PROSHOP|PIER 27|PIER27|CORAZON|VIVA|MARINI|SUNGLASSCHIC|ALBERCA|MUL|RECEPTOR|BOARWALK|BOARDWALK|FINI|CORAZN|BODEGA)',
    'PTO85':     r'(TAB|JOY|CORONA|MULTI|PUEBLITO|RECEPTORIA|AMIANI|AQUA PARK|PROSHOP|PIER 27|PIER27|CORAZON|VIVA|MARINI|SUNGLASSCHIC|ALBERCA|MUL|RECEPTOR|BOARWALK|BOARDWALK|FINI|CORAZN|BODEGA|LOS CABOS|CABOS|ALMACEN)',
    'PTOHSRD':   r'(TAB|JOY|CORONA|MULTI|PUEBLITO|RECEPTORIA|AMIANI|AQUA PARK|PROSHOP|PIER 27|PIER27|CORAZON|VIVA|MARINI|SUNGLASSCHIC|ALBERCA|MUL|RECEPTOR|BOARWALK|BOARDWALK|FINI|CORAZN|BODEGA|LOS CABOS|CABOS|ALMACEN|RIDDIM)',
    'PELTJAM':   r'(CORAL SPRINGS|OCHO RIOS|CEDIS|TAB|JOY|CORONA|MULTI|PUEBLITO|RECEPTORIA|AMIANI|AQUA PARK|PROSHOP|PIER 27|PIER27|CORAZON|VIVA|MARINI|SUNGLASSCHIC|ALBERCA|MUL|RECEPTOR|BOARWALK|BOARDWALK|FINI|CORAZN|BODEGA|LOS CABOS|CABOS|ALMACEN|RIDDIM)',
    'OHSGRLTD':  r'(CORAL SPRINGS|OCHO RIOS|CEDIS|TAB|JOY|CORONA|MULTI|PUEBLITO|RECEPTORIA|AMIANI|AQUA PARK|PROSHOP|PIER 27|PIER27|CORAZON|VIVA|MARINI|SUNGLASSCHIC|ALBERCA|MUL|RECEPTOR|BOARWALK|BOARDWALK|FINI|CORAZN|BODEGA|LOS CABOS|CABOS|ALMACEN|RIDDIM)',
    'OHSXCD':    r'(CORAL SPRINGS|OCHO RIOS|CEDIS|TAB|JOY|CORONA|MULTI|PUEBLITO|RECEPTORIA|AMIANI|AQUA PARK|PROSHOP|PIER 27|PIER27|CORAZON|VIVA|MARINI|SUNGLASSCHIC|ALBERCA|MUL|RECEPTOR|BOARWALK|BOARDWALK|FINI|CORAZN|BODEGA|LOS CABOS|CABOS|ALMACEN|RIDDIM)',
}

In [ ]:
partes = []
for empresa, keywords in EMPRESA_CONFIG.items():
    df = util_df[util_df['RAZON_SOCIAL'] == empresa].copy()
    if df.empty:
        print(f'  ⚠️  {empresa}: sin registros')
        continue
    df['HOTEL']       = df['TIENDAS'].str.split(keywords).str[0].str.strip()
    df['HOTEL']       = df['HOTEL'].str.replace(r'^[^\w]+', '', regex=True)
    df['TIPO_EQUIPO'] = df['TIENDAS'].str.extract(keywords, expand=False).fillna('OTRO')
    partes.append(df)
    print(f'  ✅ {empresa}: {len(df)} filas')

final_df = pd.concat(partes, ignore_index=True)
print(f'\n✅ Total: {len(final_df)} filas transformadas')
display(final_df[['RAZON_SOCIAL','TIENDAS','HOTEL','TIPO_EQUIPO']].head(15))

## 4️⃣ Extraer VENDOR desde MODELO
```
DELL OPTIPLEX 7050  →  DELL
LENOVO M800         →  LENOVO
```

In [ ]:
def extraer_vendor(modelo):
    if pd.isna(modelo) or str(modelo).strip() == '':
        return 'DESCONOCIDO'
    return str(modelo).split()[0]

final_df['VENDOR_NOMBRE'] = final_df['MODELO'].apply(extraer_vendor)

print('Vendors encontrados:')
display(final_df['VENDOR_NOMBRE'].value_counts().reset_index().rename(columns={'VENDOR_NOMBRE':'vendor','count':'cantidad'}))

## 5️⃣ Construir tablas con columnas exactas del schema Prisma

In [ ]:
# ── AssetState ──────────────────────────────────────────────
# Columnas: AssetStateID (PK autoincrement), Name
# Solo necesitamos un estado por defecto para la carga inicial

asset_state = pd.DataFrame([
    {'AssetStateID': 1, 'Name': DEFAULT_ASSET_STATE_NAME}
])

print(f'AssetState → {len(asset_state)} registros')
display(asset_state)

In [ ]:
# ── Company ──────────────────────────────────────────────────
# Columnas: CompanyID (PK autoincrement), Description
# RAZON_SOCIAL del CSV → Description

company = (
    final_df[['RAZON_SOCIAL']]
    .drop_duplicates()
    .reset_index(drop=True)
    .rename(columns={'RAZON_SOCIAL': 'Description'})
)
company.insert(0, 'CompanyID', company.index + 1)

print(f'Company → {len(company)} registros')
display(company)

In [ ]:
# ── Vendor ───────────────────────────────────────────────────
# Columnas: VendorID (PK autoincrement), Name

vendor = (
    final_df[['VENDOR_NOMBRE']]
    .drop_duplicates()
    .reset_index(drop=True)
    .rename(columns={'VENDOR_NOMBRE': 'Name'})
)
vendor.insert(0, 'VendorID', vendor.index + 1)

print(f'Vendor → {len(vendor)} registros')
display(vendor)

In [ ]:
# ── ProductType ──────────────────────────────────────────────
# Columnas: ProductTypeID (PK), Name, Category, Group, SubCategory
# TIPO_EQUIPO → Name
# Category / Group / SubCategory: 'HARDWARE' como valor por defecto
# (ajusta si tu sistema ya tiene categorías definidas)

product_type = (
    final_df[['TIPO_EQUIPO']]
    .drop_duplicates()
    .reset_index(drop=True)
    .rename(columns={'TIPO_EQUIPO': 'Name'})
)
product_type.insert(0, 'ProductTypeID', product_type.index + 1)
product_type['Category']    = 'HARDWARE'
product_type['Group']       = 'INFORMATICA'
product_type['SubCategory'] = 'EQUIPO DE COMPUTO'

print(f'ProductType → {len(product_type)} registros')
display(product_type)

In [ ]:
# ── Site ─────────────────────────────────────────────────────
# Columnas: SiteID (PK), Name, CompanyID (FK → Company)
# HOTEL → Name

site_raw = (
    final_df[['HOTEL', 'RAZON_SOCIAL']]
    .drop_duplicates()
    .reset_index(drop=True)
)
site_raw = site_raw.merge(company, left_on='RAZON_SOCIAL', right_on='Description', how='left')
site = site_raw[['HOTEL', 'CompanyID']].rename(columns={'HOTEL': 'Name'})
site = site[site['Name'].notna() & (site['Name'] != '')].reset_index(drop=True)
site.insert(0, 'SiteID', site.index + 1)

print(f'Site → {len(site)} registros')
display(site.head(10))

In [ ]:
# ── Asset (tabla de hechos) ───────────────────────────────────
# Columnas requeridas: AssetID, Name, VendorID, ProductTypeID,
#                      AssetState, CompanyID, SiteID
# Columnas opcionales: UserID, ParentAssetID, DepartID → NULL en carga inicial

asset = final_df.copy().reset_index(drop=True)

# Resolver FK CompanyID
asset = asset.merge(company, left_on='RAZON_SOCIAL', right_on='Description', how='left')

# Resolver FK SiteID
asset = asset.merge(
    site[['SiteID', 'Name', 'CompanyID']],
    left_on=['HOTEL', 'CompanyID'],
    right_on=['Name', 'CompanyID'],
    how='left',
    suffixes=('', '_site')
)

# Resolver FK ProductTypeID
asset = asset.merge(product_type[['ProductTypeID', 'Name']], left_on='TIPO_EQUIPO', right_on='Name', how='left', suffixes=('', '_pt'))

# Resolver FK VendorID
asset = asset.merge(vendor, left_on='VENDOR_NOMBRE', right_on='Name', how='left', suffixes=('', '_v'))

# Columnas finales exactas del schema
asset_final = pd.DataFrame({
    'Name':          asset['TIENDAS'],          # nombre del equipo/tienda
    'VendorID':      asset['VendorID'],
    'ProductTypeID': asset['ProductTypeID'],
    'AssetState':    1,                          # FK al AssetState 'Activo'
    'CompanyID':     asset['CompanyID'],
    'SiteID':        asset['SiteID'],
    'UserID':        None,                       # sin asignar en carga inicial
    'ParentAssetID': None,
    'DepartID':      None,
})
asset_final.insert(0, 'AssetID', asset_final.index + 1)

print(f'Asset → {len(asset_final)} registros')
display(asset_final.head(10))

In [ ]:
# ── AssetDetail ───────────────────────────────────────────────
# Columnas mapeadas desde el CSV:
#   MODELO        → Model
#   SO            → OperatingSystem / OSName
#   CPU           → Processor
#   RAM           → PhysicalMemory / RAM
#   HHD SSD NVME  → HDDCapacity
#   NS            → SerialNum
#   HOSTNAME      → (no hay campo directo, va en Model o NumModel)
# LastUpdateBy    → SYSTEM_USER_ID (obligatorio, no nullable)

now = datetime.now()

asset_detail = pd.DataFrame({
    'AssetID':          asset_final['AssetID'],
    'Model':            final_df['MODELO'],
    'NumModel':         final_df['HOSTNAME'],   # hostname como modelo numérico/alias
    'SerialNum':        final_df['NS'],
    'OperatingSystem':  final_df['SO'],
    'OSName':           final_df['SO'],
    'Processor':        final_df['CPU'],
    'PhysicalMemory':   final_df['RAM'],
    'RAM':              final_df['RAM'],
    'HDDCapacity':      final_df['HHD SSD NVME'],
    'LastUpdateBy':     SYSTEM_USER_ID,
    'CreatedTime':      now,
    'LastUpdateTime':   now,
})
asset_detail.insert(0, 'AssetDetailID', asset_detail.index + 1)

print(f'AssetDetail → {len(asset_detail)} registros')
display(asset_detail.head(10))

## 6️⃣ Exportar CSVs a Drive

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

tablas = {
    '1_AssetState':  asset_state,
    '2_Company':     company,
    '3_Vendor':      vendor,
    '4_ProductType': product_type,
    '5_Site':        site,
    '6_Asset':       asset_final,
    '7_AssetDetail': asset_detail,
}

print('📂 Archivos generados en:', OUTPUT_DIR)
for nombre, df in tablas.items():
    ruta = f'{OUTPUT_DIR}{nombre}.csv'
    df.to_csv(ruta, index=False, encoding='utf-8-sig')
    print(f'  ✅ {nombre}.csv — {len(df)} filas, columnas: {list(df.columns)}')

## 7️⃣ Reporte de validación
Verifica que todas las FK quedaron resueltas. Si hay nulls en las FK obligatorias, hay que corregirlos **antes** del Load real.

In [ ]:
print('=' * 50)
print('📊 REPORTE DE VALIDACIÓN')
print('=' * 50)
print(f'  AssetState únicos  : {len(asset_state)}')
print(f'  Empresas           : {len(company)}')
print(f'  Vendors            : {len(vendor)}')
print(f'  Tipos de equipo    : {len(product_type)}')
print(f'  Sites/Hoteles      : {len(site)}')
print(f'  Activos (Asset)    : {len(asset_final)}')
print(f'  Detalles           : {len(asset_detail)}')
print()

# FK obligatorias en Asset (no pueden ser null)
fks_obligatorias = ['VendorID', 'ProductTypeID', 'AssetState']
fks_opcionales   = ['CompanyID', 'SiteID']

nulos_oblig = asset_final[fks_obligatorias].isna().sum()
nulos_opc   = asset_final[fks_opcionales].isna().sum()

if nulos_oblig.sum() == 0:
    print('  ✅ FK obligatorias: todas resueltas')
else:
    print('  ❌ FK obligatorias con null (BLOQUEAN el load):')
    display(nulos_oblig[nulos_oblig > 0])

if nulos_opc.sum() == 0:
    print('  ✅ FK opcionales  : todas resueltas')
else:
    print(f'  ⚠️  FK opcionales con null (permitidas, revisar):')
    display(nulos_opc[nulos_opc > 0])

print()

# Mostrar registros con FK opcionales en null para revisión
mask_opc = asset_final[fks_opcionales].isna().any(axis=1)
if mask_opc.sum() > 0:
    print(f'  Registros sin Site o Company ({mask_opc.sum()} total):')
    display(asset_final[mask_opc][['AssetID', 'Name', 'CompanyID', 'SiteID']].head(10))

print()
if nulos_oblig.sum() == 0:
    print('  🚀 Listo para el Load real a SQL Server')
else:
    print('  🔧 Corrige los nulls obligatorios antes de cargar a la BD')
print('=' * 50)